# Company Financial Ratio Screener

This notebook pulls real financial statement data for 10 companies across four sectors — **Technology, Banking, UK Consumer, and Energy** — and compares them using four core financial ratios: Return on Equity, Current Ratio, Debt-to-Equity, and Net Profit Margin.

## Research Questions

1. Which companies show the strongest overall profitability?
2. How does financial leverage (debt) differ across sectors?
3. Which companies have the strongest short-term liquidity?
4. How does the same ratio mean different things across sectors — and where does a "one-size-fits-all" comparison break down?

Data is pulled live via the `yfinance` API, so exact figures will vary slightly depending on when this notebook is run.

In [ ]:
import yfinance as yf
import pandas as pd
from matplotlib import pyplot as plt

In [ ]:
tickers = {
    "Apple": "AAPL",
    "Microsoft": "MSFT",
    "Meta": "META",
    "Tesla": "TSLA",
    "JPMorgan": "JPM",
    "HSBC": "HSBC",
    "Barclays": "BARC.L",
    "Unilever": "ULVR.L",
    "Tesco": "TSCO.L",
    "Shell": "SHEL"
}

In [ ]:
tickers

In [ ]:
data = {}

In [ ]:
for name, ticker in tickers.items():
    company = yf.Ticker(ticker)
    data[name] = {"Balance_Sheet" : company.balance_sheet,
    "income_stmt" : company.financials}
    print("Pulled data for " + name)

In [ ]:
for name in data:
    print(name)
    print(data[name]['Balance_Sheet'].index)

In [ ]:
for name in data:
    print (name)
    print (data[name]['income_stmt'].index)

In [ ]:
for name in data:
    print ({name})
    display(data[name]['Balance_Sheet'])

In [ ]:
for name in data:
    print ({name})
    display(data[name]['income_stmt'])

### Return on Equity (ROE)

ROE measures the profit generated for each unit of shareholders' equity.

$$
\text{ROE} = \frac{\text{Net Income}}{\text{Stockholders' Equity}}
$$

**What it means:** A higher ROE generally indicates that management is generating more profit from the shareholders' capital invested in the business. However, ROE can be affected by financial leverage and unusually low equity, so it should be interpreted together with Debt-to-Equity.

In [ ]:
Companies =[]
roe_value = []
for name in data:
    print (name)
    N_Income = (data[name]['income_stmt'].loc['Net Income'].iloc[0])
    Stock_holders = (data[name]['Balance_Sheet'].loc['Stockholders Equity'].iloc[0])
    roe = N_Income / Stock_holders
    Companies.append(name)
    roe_value.append(roe)
    print(roe)

In [ ]:
plt.barh(Companies, roe_value)
plt.xlabel('Return on Equity')
plt.tight_layout()

### Current Ratio

The Current Ratio measures a company's ability to cover its short-term liabilities using its short-term assets.

$$
\text{Current Ratio} = \frac{\text{Current Assets}}{\text{Current Liabilities}}
$$

**What it means:**
- **Above 1.0:** current assets exceed current liabilities.
- **Around 1.0:** short-term assets and liabilities are broadly balanced.
- **Below 1.0:** current liabilities exceed current assets, which can indicate tighter short-term liquidity.

The analysis excludes banks because the conventional current-ratio framework is not directly comparable with a bank's balance sheet structure and liquidity model.

In [ ]:
for name in data:
    if 'Current Assets' in data[name]['Balance_Sheet'].index:
        print (name)
    else:
        print (name, "Not there")

### Current Ratio Calculation

The following cell calculates the Current Ratio for the non-bank companies and explicitly identifies the bank cases where the ratio is treated as not applicable.

**Interpretation:** This ratio is most useful when comparing companies with similar operating and working-capital structures.

In [ ]:
Current_Ratio_Companies=[]
Ratio=[]
for name in data:
    if 'Current Assets' in data[name]['Balance_Sheet'].index:
        Current_Assets = (data[name]['Balance_Sheet'].loc['Current Assets'].iloc[0])
        Current_liabilities = (data[name]['Balance_Sheet'].loc['Current Liabilities'].iloc[0])
        print ("Details for ", name)
        print ("Current Assets for the", name,":", Current_Assets)
        print ("Current Liabilities for the", name,":", Current_liabilities)
        current_ratio_results = Current_Assets / Current_liabilities
        Current_Ratio_Companies.append(name)
        Ratio.append(current_ratio_results)
    else:
        print ("      ")
        print (name, "- Current Ratio not applicable (bank)")
        print ("      ")

### Current Ratio Comparison

The chart shows short-term liquidity for the non-bank companies. Higher is not automatically better: a very high ratio can also indicate that capital is tied up in cash, receivables, or inventory rather than being used productively.

In [ ]:
plt.bar(Current_Ratio_Companies, Ratio, color='#e5ae38')
plt.xlabel('Current Ratio')
plt.grid()
plt.tight_layout()

### Debt-to-Equity Ratio

Debt-to-Equity compares interest-bearing debt with shareholders' equity.

$$
\text{Debt-to-Equity} = \frac{\text{Total Debt}}{\text{Stockholders' Equity}}
$$

**What it means:** A higher ratio indicates greater reliance on debt financing relative to equity. Higher leverage can magnify returns to shareholders when business performance is strong, but it also increases financial risk and debt-service obligations.

Sector context is essential. Banks naturally operate with substantially more leverage than most non-financial companies because deposits and other liabilities are integral to their business model.

In [ ]:
Data_Debt_to_Equity =[]
for name in data:
    Total_Debt = data[name]['Balance_Sheet'].loc['Total Debt'].iloc[0]
    Stock_holders = (data[name]['Balance_Sheet'].loc['Stockholders Equity'].iloc[0])
    Debt_to_Equity = Total_Debt / Stock_holders
    Data_Debt_to_Equity.append(Debt_to_Equity)
    print ("Debt_to_Equity of", (name), "is :", Debt_to_Equity)

### Debt-to-Equity Comparison

This chart compares financial leverage across the selected companies. The bank results should not be interpreted using the same risk benchmark as technology, consumer, or energy companies.

In [ ]:
plt.barh(Companies, Data_Debt_to_Equity, color='#9B59B6')
plt.xlabel("Debt to Equity")

### Net Profit Margin

Net Profit Margin measures how much of each unit of revenue remains as profit after all expenses, interest, and taxes.

$$
\text{Net Profit Margin} = \frac{\text{Net Income}}{\text{Total Revenue}}
$$

**What it means:** A higher margin means the company retains more profit from its sales. Differences across sectors are important because banks, technology companies, consumer businesses, and energy companies have very different business models and cost structures.

In [ ]:
Net_Profit_Margin =[]
for name in data:
    Revenue = data[name]['income_stmt'].loc['Total Revenue'].iloc[0]
    Net_income = (data[name]['income_stmt'].loc['Net Income'].iloc[0])
    Net_Profit = Net_income / Revenue
    Net_Profit_Margin.append(Net_Profit)
    print(name, "Net Profit Margin is:", Net_Profit)

In [ ]:
plt.style.available

### Net Profit Margin Comparison

The final chart ranks the companies by the proportion of revenue retained as net income. This is particularly useful for comparing the profitability profile of the four sectors represented in the dataset.

In [ ]:
plt.style.use("seaborn-v0_8-notebook")
plt.barh(Companies, Net_Profit_Margin, color='#1E1E1E')
plt.xlabel ("Net Profit Margin")
plt.tight_layout()

## Key Findings

The analysis covers four groups: **Technology (Apple, Microsoft, Meta, Tesla), Banks (JPMorgan, HSBC, Barclays), UK Consumer (Unilever, Tesco), and Energy (Shell)**. The comparisons below use the ratios calculated in this notebook.

### 1. Technology — strongest overall profitability, but with wide variation

- The technology group shows the strongest overall **ROE**, with Apple standing out at approximately **151.9%**. Microsoft, Meta, and Tesla are much lower, showing that profitability differs substantially even within the same sector.
- **Net Profit Margin** is also strong across most technology companies. Microsoft has the highest margin in the dataset at approximately **40.3%**, followed by Meta at **30.1%** and Apple at **26.9%**.
- Technology companies generally show lower **Debt-to-Equity** than the banks and UK consumer companies. Microsoft is particularly lightly leveraged at approximately **0.13**.
- Liquidity is mixed: Meta (**2.60**) and Tesla (**2.16**) have strong current ratios, Microsoft is above 1 (**1.23**), while Apple is below 1 (**0.89**).

**Overall:** Technology has the strongest combination of profitability and relatively moderate leverage, although Apple’s exceptionally high ROE should be interpreted carefully because ROE can be amplified by a low equity base.

### 2. Banks — high profitability margins but the highest leverage

- JPMorgan, HSBC, and Barclays all have relatively strong **Net Profit Margins**, ranging from approximately **24.6% to 31.4%**.
- The banks have lower **ROE** than the technology group in this dataset: JPMorgan is approximately **15.7%**, HSBC **11.2%**, and Barclays **9.2%**.
- **Debt-to-Equity is substantially higher** for the banks, particularly Barclays at approximately **2.83**, compared with most technology companies.
- **Current Ratio is not used for the banks** in this analysis because the conventional current-assets/current-liabilities framework is not directly comparable with banking balance sheets.

**Overall:** The banking sector demonstrates strong bottom-line margins but operates with materially higher financial leverage. Its ratios should therefore be interpreted using banking-sector norms rather than non-financial-company benchmarks.

### 3. UK Consumer — weaker liquidity and mixed profitability

- Unilever and Tesco show the weakest **current liquidity** among the non-bank groups. Unilever's current ratio is approximately **0.79**, while Tesco's is only **0.59**.
- Profitability is very different between the two companies. Unilever has a **Net Profit Margin of approximately 18.7%** and ROE of approximately **61.0%**, while Tesco has only **2.4%** net margin and approximately **15.6%** ROE.
- Both companies use relatively high leverage: Unilever's Debt-to-Equity is approximately **1.78**, while Tesco's is approximately **1.32**.

**Overall:** The UK consumer group has weaker liquidity than technology and Shell, and its profitability is highly dependent on the individual company. Unilever is substantially stronger than Tesco on the profitability measures used here.

### 4. Energy — moderate liquidity, lower profitability, lower leverage

- Shell has a **Current Ratio of approximately 1.30**, indicating more current assets than current liabilities.
- Its **Debt-to-Equity of approximately 0.43** is lower than the banks and UK consumer companies and is also below Apple.
- However, Shell's **Net Profit Margin is approximately 6.7%** and ROE is approximately **10.2%**, both relatively modest compared with the strongest technology and banking results.

**Overall:** Shell appears less leveraged and reasonably liquid, but its profitability ratios are lower. This highlights the capital-intensive and cyclical characteristics of the energy business.

### Sector-Level Summary

| Sector | Profitability | Liquidity | Leverage | Main takeaway |
|---|---|---|---|---|
| **Technology** | Strongest overall | Generally strong, but Apple < 1 | Moderate to low | Best overall profitability profile |
| **Banks** | Strong margins | Current Ratio not comparable | Highest | Profitable but highly leveraged by design |
| **UK Consumer** | Mixed | Weakest | High | Liquidity pressure and large company differences |
| **Energy** | Lower | Moderate | Relatively low | Lower profitability but conservative leverage |

### Final Conclusion

The ratios show that **technology companies have the strongest overall profitability profile**, while **banks generate strong net margins but rely much more heavily on leverage**. The **UK consumer companies have the weakest liquidity position**, with Tesco particularly weak on profitability, whereas **Shell combines moderate liquidity with comparatively low leverage but lower profitability**.

The most important lesson is that **financial ratios should not be compared in isolation or without sector context**. A high Debt-to-Equity ratio may be normal for a bank but more concerning for a technology company, while a low Current Ratio may be manageable for a business with strong cash generation but warrants closer attention in a company with weaker margins.